<a href="https://colab.research.google.com/github/kawastony/grok-notes-version-2/blob/main/L%3D12_qm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from itertools import product
from scipy.sparse import coo_matrix, diags
from scipy.sparse.linalg import eigsh
from scipy.linalg import svd
import json, gc, time

I2 = np.eye(2, dtype=complex)
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
z2 = np.zeros((2, 2), dtype=complex)
alpha4 = [np.block([[z2, s], [s, z2]]) for s in (sx, sy, sz)]
beta4 = np.block([[I2, z2], [z2, -I2]])
ALPHA = [np.kron(a, I2).astype(complex) for a in alpha4]
BETA = np.kron(beta4, I2).astype(complex)
MASS_B = [np.kron(beta4, t).astype(complex) for t in (sx, sy, sz)]
EYE8 = np.eye(8, dtype=complex)
G5DIAG = np.diag(np.kron(beta4, I2)).real.copy()

def hedgehog(x, y, z, cx, cy, cz, L, ww):
    rx = (x - cx) - L * int(round((x - cx) / float(L)))
    ry = (y - cy) - L * int(round((y - cy) / float(L)))
    rz = (z - cz) - L * int(round((z - cz) / float(L)))
    rr = np.sqrt(rx*rx + ry*ry + rz*rz + 1e-16)
    f = np.tanh(rr / ww)
    th = np.arccos(np.clip(rz / rr, -1.0, 1.0))
    ph = np.arctan2(ry, rx)
    hx, hy, hz = np.sin(th)*np.cos(ph), np.sin(th)*np.sin(ph), np.cos(th)
    dd = rr / np.sqrt(rr*rr + 0.08)
    hx, hy, hz = dd*hx, dd*hy, dd*hz
    n = np.sqrt(hx*hx + hy*hy + hz*hz + 1e-30)
    return f, hx/n, hy/n, hz/n

def build_sparse_H(L, defects, m0, v_list, ww, r_wilson):
    Nspin = 8
    N = Nspin * L**3
    rows, cols, data = [], [], []
    def add(i0, j0, mat):
        for s in range(8):
            for sp in range(8):
                val = mat[s, sp]
                if abs(val) > 1e-18:
                    rows.append(i0 + s); cols.append(j0 + sp); data.append(complex(val))
    base = m0 * BETA + 3 * r_wilson * EYE8
    for x, y, z in product(range(L), repeat=3):
        i0 = Nspin * (z + L * (y + L * x))
        Md = base.copy()
        for (sign, cx, cy, cz), v in zip(defects, v_list):
            f, hx, hy, hz = hedgehog(x, y, z, cx, cy, cz, L, ww)
            if f < 1e-14:
                continue
            Md = Md + sign * v * f * (hx*MASS_B[0] + hy*MASS_B[1] + hz*MASS_B[2])
        add(i0, i0, Md)
        for mu, (dx, dy, dz) in enumerate([(1,0,0), (0,1,0), (0,0,1)]):
            jp = Nspin * (((z+dz)%L) + L * (((y+dy)%L) + L * ((x+dx)%L)))
            jm = Nspin * (((z-dz)%L) + L * (((y-dy)%L) + L * ((x-dx)%L)))
            add(i0, jp, -0.5 * (r_wilson * EYE8 - ALPHA[mu]))
            add(i0, jm, -0.5 * (r_wilson * EYE8 + ALPHA[mu]))
    D = coo_matrix((data, (rows, cols)), shape=(N, N), dtype=complex).tocsr()
    G = diags(np.tile(G5DIAG, L**3), 0, shape=(N, N), dtype=complex)
    Hh = G @ D
    H = (0.5 * (Hh + Hh.getH())).tocsr()
    del D, Hh, G, rows, cols, data
    gc.collect()
    return H

def in_ball(x, y, z, c, L, rad):
    dx = min(abs(x - c[0]), L - abs(x - c[0]))
    dy = min(abs(y - c[1]), L - abs(y - c[1]))
    dz = min(abs(z - c[2]), L - abs(z - c[2]))
    return dx*dx + dy*dy + dz*dz <= rad*rad

def subspace_dS(V, L, c1, c2, cr):
    dens = np.zeros((L, L, L))
    for i in range(V.shape[1]):
        dens += np.sum(np.abs(V[:, i].reshape(L, L, L, 8))**2, axis=3)
    dens /= dens.sum() + 1e-30
    S1 = S2 = 0.0
    for x, y, z in product(range(L), repeat=3):
        if in_ball(x, y, z, c1, L, cr): S1 += dens[x, y, z]
        if in_ball(x, y, z, c2, L, cr): S2 += dens[x, y, z]
    return float(S2 - S1)

def residual_gate(H, evec, ev, rel_gate=1e-6):
    keep, rels = [], []
    for i in range(evec.shape[1]):
        rel = np.linalg.norm(H @ evec[:, i] - ev[i] * evec[:, i]) / (np.linalg.norm(evec[:, i]) + 1e-30)
        rels.append(float(rel))
        if rel <= rel_gate:
            keep.append(i)
    return keep, rels

print("operators ready")

operators ready


In [ ]:
L, d = 12, 3
m0, v0, ww, rw = 0.3, 2.0, 1.0, 1.0
k, rel_gate, cr, eps = 4, 1e-6, 2.0, 0.05
s_vals = [0.0, 0.25, 0.5, 0.75, 1.0]   # or [0.0, 0.5, 1.0] if slow

c1 = (L//2, L//2, L//2 - d//2)
c2 = (L//2, L//2, L//2 + (d - d//2))
defects = [(+1, *c1), (-1, *c2)]
print("cores", c1, c2, "N", 8*L**3)

rows = []
V_store = []

for s in s_vals:
    t0 = time.time()
    v1 = v0 * (1 + s * eps)
    print(f"\n=== s={s:.2f} v1={v1:.4f} ===")
    H = build_sparse_H(L, defects, m0, [v1, v0], ww, rw)
    print("  nnz", H.nnz, "eigsh...")
    ev, evec = eigsh(H, k=k, sigma=0.0, which="LM", maxiter=15000, tol=1e-8)
    keep, rels = residual_gate(H, evec, ev, rel_gate)
    print("  rels", np.round(rels, 3))
    if len(keep) < 2:
        print("  WARNING: few modes passed gate; loosening to 1e-4")
        keep, rels = residual_gate(H, evec, ev, 1e-4)
    ev, evec = ev[keep], evec[:, keep]
    # phase fix
    for i in range(evec.shape[1]):
        evec[:, i] *= np.exp(-1j * np.angle(evec[np.argmax(np.abs(evec[:, i])), i]))
    dS = subspace_dS(evec, L, c1, c2, cr)
    dt = time.time() - t0
    print("  |λ|", np.round(np.abs(ev), 6), "dS", dS, f"({dt:.0f}s)")
    rows.append(dict(s=float(s), abs_lam=np.abs(ev).tolist(),
                     lam=ev.real.tolist(), dS=dS, rels=rels, n=len(keep), dt=dt))
    V_store.append(evec.copy())
    del H
    gc.collect()

# Projector continuity (Procrustes)
print("\n=== PT Tr(PQ) ===")
Tr_list = []
for i in range(len(V_store) - 1):
    n = min(V_store[i].shape[1], V_store[i+1].shape[1])
    A, B = V_store[i][:, :n], V_store[i+1][:, :n]
    W = A.conj().T @ B
    U, S, Vh = svd(W, full_matrices=False)
    Bal = B @ (U @ Vh).conj().T
    tr = float(np.real(np.linalg.norm(A.conj().T @ Bal)**2))
    Tr_list.append(dict(s_from=s_vals[i], s_to=s_vals[i+1], TrPQ=tr, S=S.tolist()))
    print(f"  {s_vals[i]:.2f}->{s_vals[i+1]:.2f} TrPQ={tr:.3f} S={np.round(S, 3)}")

G_sub = (rows[-1]["dS"] - rows[0]["dS"]) / eps
print("\nG_subspace", G_sub)
print("dS path", [r["dS"] for r in rows])

out = dict(L=12, d=3, eps=eps, s_vals=s_vals, scan=rows, PT=Tr_list, G_subspace=G_sub)
with open("L12_d3_path.json", "w") as f:
    json.dump(out, f, indent=2)
print("saved L12_d3_path.json")

cores (6, 6, 5) (6, 6, 8) N 13824

=== s=0.00 v1=2.0000 ===
  nnz 193512 eigsh...
